# LiteLLM: Provider Routing and Fallback (2026)

Goal: show a lightweight provider abstraction and fallback policy with runtime-safe guards.


In [ ]:
import os

try:
    from litellm import completion
    HAS_LITELLM = True
except Exception as exc:
    HAS_LITELLM = False
    LITELLM_IMPORT_ERROR = str(exc)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(f"litellm available: {HAS_LITELLM}")
print(f"OPENAI_API_KEY set: {bool(OPENAI_API_KEY)}")


## Simple request
Runs only when `litellm` and `OPENAI_API_KEY` are available.


In [ ]:
if not HAS_LITELLM:
    print(f"Skipping: litellm import failed ({LITELLM_IMPORT_ERROR}).")
elif not OPENAI_API_KEY:
    print("Skipping: OPENAI_API_KEY is not set.")
else:
    resp = completion(
        model="openai/gpt-4.1-mini",
        messages=[{"role": "user", "content": "Return exactly: LITELLM_OK"}],
        api_key=OPENAI_API_KEY,
        timeout=30,
    )
    text = resp.choices[0].message.content.strip()
    print(text)
    assert isinstance(text, str) and len(text) > 0


## Fallback policy sketch
Try primary model first, then fallback model if request fails.


In [ ]:
def call_with_fallback(prompt: str, models: list[str]) -> str:
    last_err = None
    for model in models:
        try:
            resp = completion(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                api_key=OPENAI_API_KEY,
                timeout=30,
            )
            return resp.choices[0].message.content.strip()
        except Exception as exc:
            last_err = exc
            print(f"fallback: {model} failed -> {exc}")
    raise RuntimeError(f"All models failed: {last_err}")

if HAS_LITELLM and OPENAI_API_KEY:
    out = call_with_fallback(
        "In one short sentence, why do fallbacks matter?",
        ["openai/gpt-4.1-mini", "openai/gpt-4.1-nano"],
    )
    print(out)
    assert out
else:
    print("Skipping fallback demo: prerequisites missing.")
